# Application Scenario 3: Land Cover Segmentation

This scenario exports the output to the `output` directory.\
Please clear this directory between pipeline executions.

## Download the resources

- configuration file
- input directory containing the GeoJSON file

In [ ]:
!curl -sL https://raw.githubusercontent.com/geospaitial-lab/aviary-ACM-SIGSPATIAL-2026/main/application_scenarios/3/config.yaml -o config.yaml
!mkdir input
!curl -sL https://raw.githubusercontent.com/geospaitial-lab/aviary-ACM-SIGSPATIAL-2026/main/application_scenarios/3/input/boundary.geojson -o input/boundary.geojson

## Installation

### With `pip`

In [ ]:
!pip install geospaitial-lab-aviary[cli] ipywidgets onnxruntime-gpu

### With `uv`

In [ ]:
!uv pip install geospaitial-lab-aviary[cli] ipywidgets onnxruntime-gpu

---

## Python API

### Imports

In [ ]:
from pathlib import Path
import warnings

import aviary
import aviary.pipeline
import aviary.tile
import aviary.utils
import aviary.vector
import geopandas as gpd

from my_plugin import Device, LandCoverModel

### Check the version

In [ ]:
print(aviary.__version__)

### Suppress experimental warnings

Some components used in this scenario are experimental.

In [ ]:
warnings.filterwarnings('ignore', category=aviary.AviaryExperimentalWarning)

### Define the `Grid`

[Docs](https://geospaitial-lab.github.io/aviary/api_reference/core/grid)

In [ ]:
gdf = gpd.read_file('input/boundary.geojson')
gdf = gdf.to_crs('EPSG:25832')

grid = aviary.Grid.from_gdf(
    gdf=gdf,
    tile_size=100,
)

grid = grid.chunk(num_chunks=500)[93]

### Define the `TileFetcher`

[Docs](https://geospaitial-lab.github.io/aviary/api_reference/tile/tile_fetcher/tile_fetcher)

In [ ]:
wms_fetcher = aviary.tile.WMSFetcher(
    url='https://www.wms.nrw.de/geobasis/wms_nw_dop',
    version=aviary.WMSVersion.V1_3_0,
    layer='nw_dop_rgb',
    epsg_code=25832,
    response_format='image/png',
    channel_names=['r', 'g', 'b'],
    tile_size=100,
    ground_sampling_distance=.2,
    buffer_size=20,
)

### Define the `TilesProcessor`

[Docs](https://geospaitial-lab.github.io/aviary/api_reference/tile/tiles_processor/tiles_processor)

In [ ]:
standardize_processor_r = aviary.tile.StandardizeProcessor(
    channel_name='r',
    mean_value=105.66,
    std_value=52.23,
)

standardize_processor_g = aviary.tile.StandardizeProcessor(
    channel_name='g',
    mean_value=111.35,
    std_value=45.62,
)

standardize_processor_b = aviary.tile.StandardizeProcessor(
    channel_name='b',
    mean_value=102.18,
    std_value=44.30,
)

land_cover_model = LandCoverModel(
    r_channel_name='r',
    g_channel_name='g',
    b_channel_name='b',
    land_cover_channel_name='land_cover',
    device=Device.CPU,
)

sieve_processor = aviary.tile.SieveProcessor(
    channel_name='land_cover',
    threshold=25,
)

remove_buffer_processor = aviary.tile.RemoveBufferProcessor()

vectorize_processor = aviary.tile.VectorizeProcessor(
    channel_name='land_cover',
    field='category'
)

vector_exporter = aviary.tile.VectorExporter(
    channel_name='land_cover',
    epsg_code=25832,
    path=Path('output/land_cover.gpkg'),
)

grid_exporter = aviary.tile.GridExporter(
    path=Path('output/processed.json'),
)

composite_processor = aviary.tile.SequentialCompositeProcessor(
    tiles_processors=[
        standardize_processor_r,
        standardize_processor_g,
        standardize_processor_b,
        land_cover_model,
        sieve_processor,
        remove_buffer_processor,
        vectorize_processor,
        vector_exporter,
        grid_exporter,
    ],
)

### Define the `TilePipeline`

[Docs](https://geospaitial-lab.github.io/aviary/api_reference/pipeline/pipeline)

In [ ]:
tile_pipeline = aviary.pipeline.TilePipeline(
    grid=grid,
    tile_fetcher=wms_fetcher,
    tiles_processor=composite_processor,
    tile_loader_batch_size=2,
    tile_loader_max_num_threads=None,
    tile_loader_num_prefetched_tiles=0,
    show_progress=True,
)

### Define the `VectorLoader`

[Docs](https://geospaitial-lab.github.io/aviary/api_reference/vector/vector_loader/vector_loader)

In [ ]:
gpkg_loader = aviary.vector.GPKGLoader(
    path=Path('output/land_cover.gpkg'),
    epsg_code=25832,
    layer_name='land_cover',
)

geojson_loader = aviary.vector.GeoJSONLoader(
    path=Path('input/boundary.geojson'),
    epsg_code=25832,
    layer_name='boundary',
)

composite_loader = aviary.vector.CompositeLoader(
    vector_loaders=[gpkg_loader, geojson_loader],
)

### Define the `VectorProcessor`

[Docs](https://geospaitial-lab.github.io/aviary/api_reference/vector/vector_processor/vector_processor)

In [ ]:
clip_processor = aviary.vector.ClipProcessor(
    layer_name='land_cover',
    mask_layer_name='boundary',
)

map_field_processor = aviary.vector.MapFieldProcessor(
    layer_name='land_cover',
    field='category',
    mapping={
        0: 'Building',
        1: 'Greenhouse',
        2: 'Swimming pool',
        3: 'Impervious surface',
        4: 'Pervious surface',
        5: 'Bare soil',
        6: 'Water',
        7: 'Snow',
        8: 'Herbaceous vegetation',
        9: 'Agricultural land',
        10: 'Plowed land',
        11: 'Vineyard',
        12: 'Deciduous',
        13: 'Coniferous',
        14: 'Brushwood',
        15: 'Clear cut',
        16: 'Ligneous',
        17: 'Mixed',
        18: 'Undefined',
    }
)

vector_exporter_ = aviary.vector.VectorExporter(
    layer_name='land_cover',
    epsg_code=25832,
    path=Path('output/land_cover.gpkg'),
)

composite_processor_ = aviary.vector.SequentialCompositeProcessor(
    vector_processors=[
        clip_processor,
        map_field_processor,
        vector_exporter_,
    ]
)

### Define the `VectorPipeline`

[Docs](https://geospaitial-lab.github.io/aviary/api_reference/pipeline/pipeline)

In [ ]:
vector_pipeline = aviary.pipeline.VectorPipeline(
    vector_loader=composite_loader,
    vector_processor=composite_processor_,
)

### Define the `CompositePipeline`

[Docs](https://geospaitial-lab.github.io/aviary/api_reference/pipeline/pipeline)

In [ ]:
composite_pipeline = aviary.pipeline.CompositePipeline(
    pipelines=[tile_pipeline, vector_pipeline]
)

### Execute the `CompositePipeline`

In [ ]:
composite_pipeline()

---

## CLI

### Check the version

In [ ]:
!aviary --version

### Suppress experimental warnings

Some components used in this scenario are experimental.

In [ ]:
%env AVIARY_EXPERIMENTAL_WARNINGS=false

### Validate the configuration file

[Docs](https://geospaitial-lab.github.io/aviary/cli_reference/aviary_pipeline/pipeline_validate)

In [ ]:
!aviary pipeline validate config.yaml

### Set the path to the log file and the log level

In [ ]:
%env AVIARY_LOG_PATH=output/application_scenario_3.log
%env AVIARY_LOG_LEVEL=trace

### Execute the `CompositePipeline`

[Docs](https://geospaitial-lab.github.io/aviary/cli_reference/aviary_pipeline/pipeline_run)

In [ ]:
!aviary pipeline run config.yaml